# Characterization Process (PMDCo): from data entry to RDF

This notebook shows how to record a characterization experiment with full
provenance context and convert it into a standardised, machine-readable RDF
graph following the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

This schema is the **guided intake layer** for characterization experiments.
Three provenance fields are mandatory — no record is complete without them:

| Field | What it captures | Ontology property |
|---|---|---|
| `operator_iri` | Who conducted the experiment | `prov:wasAssociatedWith` |
| `device_iri` | Which instrument was used | `schema:instrument` |
| `specimen_iri` | What material was characterised | `obi:has_specified_input` |

The optional `step_reference` field links this provenance record to a detailed
result node elsewhere in the graph (e.g. a `tto:TensileTest` recorded with
`characterization/step/tensile-test/TTO/`).


## What the notebook does

```
example.input.json
  │  process name, operator IRI, device IRI, specimen IRI, date, conditions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms operator, device, and specimen are all present
SPARQL inspect    →  shows the provenance links recorded in the graph
```


## Prerequisites: register your entities first

Before filling in this form, the following nodes should already exist in the
knowledge graph with stable IRIs:

- **Specimen** — register with [`specimen/PMDCo/`](../../../specimen/PMDCo/README.md)
- **Measurement device** — register with [`measurement-device/PMDCo/`](../../../measurement-device/PMDCo/README.md)
- **Expert / operator** — register with [`expertise/`](../../../expertise/README.md)

Copy the IRI from each registered node and paste it into the three required fields.


## Environment setup

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # characterization/process/PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))

Schema : characterization/process/PMDCo


## Step 1: Describe your experiment

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `process_name` | **yes** | Human-readable name for this experiment |
| `operator_iri` | **yes** | IRI of the person who conducted the experiment |
| `device_iri` | **yes** | IRI of the measurement device used |
| `specimen_iri` | **yes** | IRI of the specimen being characterised |
| `date` | no | Experiment date/time (ISO 8601) |
| `step_reference` | no | IRI of the detailed result node (e.g. tto:TensileTest) |
| `conditions` | no | List of quantitative test parameters (name, value, unit) |
| `preceded_by` | no | IRIs of preceding characterization processes |
| `process_id` | no | Custom IRI slug; auto-derived from `process_name` if omitted |

**What is an IRI?**  
An IRI is a web address that uniquely identifies a node in the knowledge graph.
Copy the IRI from the node registered for your specimen, device, and operator.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "process_name": "Tensile test 316L batch 1 \u2014 QA run",
  "operator_iri": "https://example.org/people/jane-doe",
  "device_iri": "https://example.org/devices/zwick-z250-1",
  "specimen_iri": "https://example.org/specimens/316L-tensile-bar-1",
  "date": "2024-11-15T09:30:00",
  "step_reference": "https://example.org/characterization/tensile-test-316L-batch-1",
  "conditions": [
    {
      "name": "Test Standard",
      "unit": "ISO 6892-1"
    },
    {
      "name": "Strain Rate",
      "value": 0.00025,
      "unit": "1/s"
    },
    {
      "name": "Temperature",
      "value": 23,
      "unit": "\u00b0C"
    }
  ]
}


## Step 2: Convert to OO-LD

Transforms your plain JSON into a structured OO-LD document.
Every field is linked to a precise ontology term:

| Simplified field | OO-LD field | Ontology property |
|---|---|---|
| `operator_iri` | `operator_iri` | `prov:wasAssociatedWith` |
| `device_iri` | `device_iri` | `schema:instrument` |
| `specimen_iri` | `has_specified_input` | `obi:OBI_0000293` |
| `step_reference` | `step_reference` | `dcterms:references` |
| `date` | `date` | `dcterms:date` |

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/process/PMDCo/#v1.0.0",
  "type": "obi:0000070",
  "id": "char-process-tensile-test-316l-batch-1-qa-run",
  "label": "Tensile test 316L batch 1 \u2014 QA run",
  "operator_iri": "https://example.org/people/jane-doe",
  "device_iri": "https://example.org/devices/zwick-z250-1",
  "has_specified_input": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "date": "2024-11-15T09:30:00",
  "step_reference": "https://example.org/characterization/tensile-test-316L-batch-1",
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "char-process-tensile-test-316l-batch-1-qa-run-condition-1",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "char-process-tensile-test-316l-batch-1-qa-run-condition-2",
      "parameter_label": "Strain Rate",
      "parameter_value":

## Step 3: Convert to RDF

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 22 triples.



@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix prov1: <https://www.w3.org/ns/prov#> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/char-process-tensile-test-316l-batch-1-qa-run> a obo:OBI_0000070 ;
    rdfs:label "Tensile test 316L batch 1 — QA run" ;
    obo:OBI_0000293 <https://example.org/specimens/316L-tensile-bar-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/process/PMDCo/#v1.0.0> ;
    dcterms:date "2024-11-15T09:30:00"^^xsd:dateTime ;
    dcterms:references <https://example.org/characterization/tensile-test-316L-batch-1> ;
    schema:instrument <https://example.org/devices/zwick-z250-1> ;
    pmdco:PMD_0000016 <https://w3id.org/pmd

In [6]:
out_ttl = HERE / "output_characterization_process.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_characterization_process.ttl


## Step 4: Validate against the SHACL shape

The shape file `specs/shape.ttl` checks that:

- The process has exactly one `rdfs:label`.
- `prov:wasAssociatedWith` (operator) is present and is an IRI.
- `schema:instrument` (device) is present and is an IRI.
- `obi:has_specified_input` (specimen) is present and is an IRI.
- `dcterms:references` (step result), if given, is an IRI.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the provenance graph

Confirm the three mandatory provenance links were recorded correctly.

In [8]:
OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
PROV = rdflib.Namespace("https://www.w3.org/ns/prov#")
SCH  = rdflib.Namespace("https://schema.org/")
DCT  = rdflib.Namespace("http://purl.org/dc/terms/")

proc_iri  = next(flat.subjects(rdflib.RDF.type, OBI["0000070"]))
label     = flat.value(proc_iri, rdflib.RDFS.label)
operator  = flat.value(proc_iri, PROV.wasAssociatedWith)
device    = flat.value(proc_iri, SCH.instrument)
specimen  = flat.value(proc_iri, OBI["0000293"])
step_ref  = flat.value(proc_iri, DCT.references)

print(f"Process  : {proc_iri}")
print(f"Label    : {label}")
print(f"Operator : {operator}")
print(f"Device   : {device}")
print(f"Specimen : {specimen}")
print(f"Step ref : {step_ref or '(none)'}")

Process  : https://w3id.org/pmd/co/test/char-process-tensile-test-316l-batch-1-qa-run
Label    : Tensile test 316L batch 1 — QA run
Operator : https://example.org/people/jane-doe
Device   : https://example.org/devices/zwick-z250-1
Specimen : https://example.org/specimens/316L-tensile-bar-1
Step ref : https://example.org/characterization/tensile-test-316L-batch-1


In [9]:
SPARQL_CONDITIONS = """
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?name ?value ?unit
WHERE {
  ?proc a <http://purl.obolibrary.org/obo/OBI_0000070> ;
        pmd:PMD_0000016 ?cond .
  ?cond rdfs:label ?name .
  OPTIONAL { ?cond qudt:value   ?value . }
  OPTIONAL { ?cond qudt:hasUnit ?unit  . }
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL_CONDITIONS))
if rows:
    print(f"{'Condition':<25}  {'Value':>10}  Unit")
    print("-" * 50)
    for r in rows:
        val  = f"{float(r.value):>10.4g}" if r.value else f"{'':>10}"
        unit = str(r.unit) if r.unit else ""
        print(f"{str(r.name):<25}  {val}  {unit}")
else:
    print("No conditions recorded.")

Condition                       Value  Unit
--------------------------------------------------
Strain Rate                   0.00025  1/s
Temperature                        23  °C
Test Standard                          ISO 6892-1


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with operator, device, specimen IRIs and optional metadata |
| 2 | The data is converted to an OO-LD document with ontology-mapped properties |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated — missing operator, device, or specimen causes a SHACL violation |
| 5 | Key provenance facts are inspected from the graph |


## Typical workflow

```
1. Register specimen      → specimen/PMDCo/             → get specimen IRI
2. Register device        → measurement-device/PMDCo/   → get device IRI
3. Register operator      → expertise/                  → get operator IRI
4. Record this process    → characterization/process/PMDCo/  (this schema)
5. Record detailed result → characterization/step/tensile-test/TTO/
6. Link step result       → add step_reference IRI to this record
```


## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md)
- [Schema format reference](../../../docs/3_schema-format.md)
- [Measurement Device schema](../../../measurement-device/PMDCo/README.md)
- [Tensile Test schema](../../step/tensile-test/TTO/README.md)
- [Specimen schema](../../../specimen/PMDCo/README.md)
- [Expertise schema](../../../expertise/README.md)
- [PMDCo core ontology](https://github.com/materialdigital/core-ontology)
- [PROV-O: prov:wasAssociatedWith](https://www.w3.org/TR/prov-o/#wasAssociatedWith)